--

### [Cell 1] 라이브러리 임포트 및 기본 클래스 정의

첫 번째 셀에는 도구 명세(ToolSpec)를 정의하기 위한 기본 클래스와 유효성 검사 로직을 넣습니다.

In [1]:
# =========================================================
# 1) Tool 정의를 위한 기본 클래스 및 유효성 검사
# =========================================================
from dataclasses import dataclass
from typing import Any, Callable, Dict, List
from pprint import pprint
import time

@dataclass
class ToolSpec:
    name: str
    description: str
    input_schema: Dict[str, Any]
    func: Callable[[Dict[str, Any]], Dict[str, Any]]

class ToolValidationError(Exception):
    pass

def validate_args(schema: Dict[str, Any], args: Dict[str, Any]) -> None:
    """단순한 JSON-Schema 스타일 검사기"""
    properties = schema.get("properties", {})
    required = schema.get("required", [])

    missing = [key for key in required if key not in args]
    if missing:
        raise ToolValidationError(f"Missing required fields: {missing}")

    for key, value in args.items():
        if key not in properties:
            raise ToolValidationError(f"Unknown field: {key}")

### [Cell 2] Tool Registry 정의

도구들을 등록하고 관리하는 레지스트리 클래스입니다. 에이전트는 이 레지스트리를 통해 사용 가능한 도구를 탐색합니다.

In [2]:
# =========================================================
# 2) Tool Registry: 에이전트가 사용할 도구들을 모아두는 곳
# =========================================================
class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, ToolSpec] = {}

    def register(self, name: str, description: str, input_schema: Dict[str, Any]):
        def decorator(func: Callable):
            self.tools[name] = ToolSpec(name, description, input_schema, func)
            return func
        return decorator

    def call(self, name: str, args: Dict[str, Any]) -> Dict[str, Any]:
        if name not in self.tools:
            return {"error": f"Tool '{name}' not found."}

        tool = self.tools[name]
        try:
            validate_args(tool.input_schema, args)
            return tool.func(args)
        except Exception as e:
            return {"error": str(e)}

    def describe(self):
        for name, spec in self.tools.items():
            print(f"- [{name}]: {spec.description}")

registry = ToolRegistry()

### [Cell 3] 가상의 외부 시스템 도구 등록

에이전트가 제어할 대상(가상 BMS, 날씨 API 등)을 도구 형태로 등록합니다.

In [3]:
# =========================================================
# 3) 가상의 외부 시스템 도구들 등록
# =========================================================

# 가상 상태 데이터
mock_db = {
    "zone_a": {"fan_speed": "low", "damper_open_pct": 20},
    "zone_b": {"fan_speed": "high", "damper_open_pct": 80}
}

@registry.register(
    name="get_zone_state",
    description="특정 구역(zone)의 현재 팬 속도와 댐퍼 개도율을 조회합니다.",
    input_schema={
        "type": "object",
        "properties": {"zone_id": {"type": "string"}},
        "required": ["zone_id"]
    }
)
def get_zone_state(args: Dict[str, Any]):
    zone_id = args["zone_id"]
    return {"status": "success", "content": mock_db.get(zone_id, "Unknown Zone")}

@registry.register(
    name="control_hvac",
    description="팬 속도나 댐퍼 개도율을 직접 제어합니다.",
    input_schema={
        "type": "object",
        "properties": {
            "zone_id": {"type": "string"},
            "fan_speed": {"type": "string"},
            "damper_open_pct": {"type": "integer"}
        },
        "required": ["zone_id"]
    }
)
def control_hvac(args: Dict[str, Any]):
    zone_id = args["zone_id"]
    if zone_id in mock_db:
        if "fan_speed" in args: mock_db[zone_id]["fan_speed"] = args["fan_speed"]
        if "damper_open_pct" in args: mock_db[zone_id]["damper_open_pct"] = args["damper_open_pct"]
        return {"status": "success", "message": f"{zone_id} 제어 완료"}
    return {"status": "error", "message": "Zone not found"}

@registry.register(
    name="get_weather",
    description="가상의 외부 기상 정보를 가져옵니다.",
    input_schema={
        "type": "object",
        "properties": {"location": {"type": "string"}},
        "required": ["location"]
    }
)
def get_weather(args: Dict[str, Any]):
    # 교육용 고정 데이터 반환
    return {"temperature": 28.5, "humidity": 65, "condition": "Hot"}

### [Cell 4] 에이전트 실행 로직 (Main 루프)

사용자의 요청을 분석하고 도구를 호출하는 시뮬레이션 과정입니다.

In [4]:
# =========================================================
# 4) Agent 실행 시뮬레이션
# =========================================================
def run_agent(user_request: str, zone_id: str, location: str):
    print(f"USER REQUEST: {user_request}\n")

    # STEP 1: 관찰 (Observation)
    print("STEP 1) OBSERVATION")
    obs_state = registry.call("get_zone_state", {"zone_id": zone_id})
    obs_weather = registry.call("get_weather", {"location": location})
    pprint(obs_state)
    pprint(obs_weather)

    # STEP 2: 판단 (Planning - 여기서는 시뮬레이션으로 고정된 판단 로직 사용)
    # 실제 환경에서는 LLM이 이 부분을 담당합니다.
    print("\nSTEP 2) PLANNING")
    plan = {
        "reasons": ["실외 온도가 높음", "현재 팬 속도가 낮음"],
        "actions": [{"tool": "control_hvac", "args": {"zone_id": zone_id, "fan_speed": "high"}}]
    }
    pprint(plan)

    # STEP 3: 실행 (Execution)
    print("\nSTEP 3) EXECUTION")
    for action in plan["actions"]:
        result = registry.call(action["tool"], action["args"])
        pprint(result)

    # STEP 4: 최종 확인
    print("\nSTEP 4) FINAL VERIFICATION")
    final_state = registry.call("get_zone_state", {"zone_id": zone_id})
    pprint(final_state)

### [Cell 5] 실행 테스트

실제로 에이전트를 실행하여 결과를 확인합니다.

In [5]:
if __name__ == "__main__":
    print("### 등록된 도구 목록 ###")
    registry.describe()
    print("\n" + "="*50 + "\n")

    run_agent(
        user_request="zone_a의 상태를 보고 너무 더우면 팬 속도를 높여줘.",
        zone_id="zone_a",
        location="Seoul"
    )

### 등록된 도구 목록 ###
- [get_zone_state]: 특정 구역(zone)의 현재 팬 속도와 댐퍼 개도율을 조회합니다.
- [control_hvac]: 팬 속도나 댐퍼 개도율을 직접 제어합니다.
- [get_weather]: 가상의 외부 기상 정보를 가져옵니다.


USER REQUEST: zone_a의 상태를 보고 너무 더우면 팬 속도를 높여줘.

STEP 1) OBSERVATION
{'content': {'damper_open_pct': 20, 'fan_speed': 'low'}, 'status': 'success'}
{'condition': 'Hot', 'humidity': 65, 'temperature': 28.5}

STEP 2) PLANNING
{'actions': [{'args': {'fan_speed': 'high', 'zone_id': 'zone_a'},
              'tool': 'control_hvac'}],
 'reasons': ['실외 온도가 높음', '현재 팬 속도가 낮음']}

STEP 3) EXECUTION
{'message': 'zone_a 제어 완료', 'status': 'success'}

STEP 4) FINAL VERIFICATION
{'content': {'damper_open_pct': 20, 'fan_speed': 'high'}, 'status': 'success'}


---

### 💡 코드의 핵심 포인트

* **추상화:** 에이전트는 외부 시스템의 복잡한 API를 직접 호출하지 않고, `registry.call()`이라는 표준화된 인터페이스만 사용합니다.
* **CBO 스타일 제어:** 상태 정보와 날씨 정보를 '도구'를 통해 수집한 뒤, 그 조건(Condition)에 맞춰 제어 명령을 내리는 구조를 보여줍니다.
* **확장성:** 새로운 도구가 필요하면 `@registry.register` 데코레이터를 사용하여 함수를 정의하기만 하면 에이전트가 즉시 사용할 수 있게 됩니다.